# 🦈 Shark Attacks — Exploratory Data Analysis
**Ironhack Data Analytics Bootcamp | Week 2 Project**
**Team:** Diana Carolina Yule Burbano & Irene Fafian

---

### Project Overview
This notebook is the **single merged analysis file** combining both team members' work.

| Section | Content | Contributor |
|---------|---------|-------------|
| 1 | Setup & imports | Both |
| 2 | Dataset import | Both |
| 3 | Initial exploration | Both |
| 4.1–4.2 | Column dropping & naming | Irene |
| 4.3 | String cleaning (Country, State, Location, Fatality) | Irene |
| 4.3 | Activity & Species unification | Irene |
| 4.4 | Type column standardization | Irene |
| 4.5 | Date column — multi-pass parsing | Diana |
| 4.6–4.8 | Deduplication & placeholders | Both |
| 5–8 | EDA, hypothesis validation, conclusions | 📌 Pending |

> **Data source:** [Global Shark Attack File — sharkattackfile.net](https://www.sharkattackfile.net/incidentlog.htm)
> **Module:** All reusable functions are also consolidated in `shark.py` for clean import.


---
## 📋 About the Dataset

The Global Shark Attack File (GSAF) logs shark incidents worldwide. Entries are categorized as follows:

| Category | Description |
|---|---|
| **Unprovoked** | Shark initiated contact without human provocation |
| **Provoked** | Human drew "first blood" (spearing, hooking, capturing) |
| **Watercraft** | Boat bitten or rammed; hooked/netted cases are classed as provoked |
| **Sea Disaster** | Incidents during maritime or aviation accidents |
| **Questionable** | Insufficient data to confirm shark involvement |


---
## 1️⃣ Setup — Imports & Configuration


In [ ]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)

print('✅ Libraries loaded successfully')


---
## 2️⃣ Import Dataset

The file `GSAF5.xls` is stored in the **same folder** as this notebook.
Relative path ensures the notebook runs on any machine that clones the repo.


In [ ]:
file_location = 'GSAF5.xls'

shark_df = pd.read_excel(file_location)          # raw — never modify this
shark_clean = shark_df.copy()                    # working copy

print(f'✅ Raw dataset: {shark_df.shape[0]:,} rows × {shark_df.shape[1]} columns')
shark_clean.head(3)


---
## 3️⃣ Initial Data Exploration

Before cleaning, inspect the raw dataset — structure, column types, completeness.


In [ ]:
# First 5 rows
shark_df.head()


In [ ]:
# Data types and non-null counts per column
shark_df.info()


In [ ]:
# Descriptive statistics for categorical columns
shark_df.describe(include='object')


### 🔍 Issues Identified

| Column | Issue | Action |
|---|---|---|
| `Date` | Multiple formats, strings, noise words | Multi-pass parsing → year_ext, month_ext, Season |
| `Year` | Float instead of integer | Convert to Int64 (📌 pending) |
| `Age` | Mixed formats (`?`, `30+`) | Drop — not in business case |
| `Type` | Typos (`UNprovoked`, `Boatomg`) | Map to 5 official GSAF categories |
| `Sex` | 11 unique values | Drop — not in business case |
| `Country` | Punctuation, ambiguous names, geographic entries | String clean + manual mapping |
| `State`, `Location` | Whitespace, casing inconsistencies | Strip + uppercase |
| `Activity` | ~700 unique raw values | Keyword map → 10 standard categories |
| `Species` | ~300 unique raw values | Keyword map → 15 named species |
| `Fatality` | Erroneous values (M, F, NQ, 2017) | Strip + flag |
| `Case Number.1`, `original order` | Redundant / no analytical value | Drop |
| `Unnamed: 21`, `Unnamed: 22` | 1–2 non-null values each | Drop |
| `Name`, `Source`, `pdf`, `href*` | Metadata, not for analysis | Drop |


---
## 4️⃣ Data Cleaning

All cleaning performed step-by-step with reusable functions defined inline.
The same functions are also consolidated in `shark.py` for modular import.

**Cleaning chapters:**
- 4.1 — Drop irrelevant columns
- 4.2 — Standardize column names
- 4.3 — String cleaning: Country, State, Location, Fatality, Activity, Species
- 4.4 — Type column standardization
- 4.5 — Date column: multi-pass parsing → year_ext, month_ext, Season *(Diana)*
- 4.6 — Deduplication check
- 4.7 — 📌 Placeholder: Year dtype fix
- 4.8 — 📌 Placeholder: Additional transformations


---
### 4.1 Drop Irrelevant Columns *(Irene)*

Columns removed based on D1-PLAN.docx YES/NO/MAYBE classification.
Dropped: `Age`, `Name`, `Sex`, `Source`, `Injury`, `pdf`, `href formula`, `href`,
`original order`, `Unnamed: 21`, `Unnamed: 22`.


In [ ]:
# Drop columns with no analytical value for our business case
cols_to_drop = [
    'Age', 'Name', 'Sex', 'Source', 'Injury', 'pdf',
    'href formula', 'href', 'original order', 'Unnamed: 21', 'Unnamed: 22'
]
shark_clean = shark_clean.drop(columns=cols_to_drop, errors='ignore')

print(f'✅ Columns after dropping: {shark_clean.shape[1]}')
print(list(shark_clean.columns))


---
### 4.1b Check Case Number — useful as index? *(Irene)*


In [ ]:
# Check duplicates on ID columns before deciding to drop them
print('Duplicates in Case Number:', shark_clean.duplicated(subset=['Case Number']).sum())
print('Duplicates in Case Number.1:', shark_clean.duplicated(subset=['Case Number.1']).sum())

# Both are non-unique — not suitable as primary key; drop them
shark_clean = shark_clean.drop(columns=['Case Number', 'Case Number.1'], errors='ignore')
print(f'✅ Remaining columns: {list(shark_clean.columns)}')


---
### 4.2 Standardize Column Names *(Irene)*

Strip whitespace, capitalize first letter. Rename ambiguous `Fatal y/n` → `Fatality`.


In [ ]:
def clean_col_name(col):
    """Strip whitespace and capitalize first letter of each column name."""
    return col.strip().capitalize()

shark_clean.columns = shark_clean.columns.map(clean_col_name)

# Rename ambiguous column
shark_clean.rename(columns={'Fatal y/n': 'Fatality'}, inplace=True)

print('✅ Column names standardized:')
print(list(shark_clean.columns))


---
### 4.3 String Cleaning *(Irene)*

**Technique:** strip whitespace, uppercase normalization, keyword-based category unification.
Helper functions defined once and reused across all string columns.


In [ ]:
def clean_string(value):
    """Strip whitespace and convert to uppercase. Returns None for nulls."""
    return value if pd.isnull(value) else str(value).strip().upper()

def find_weird_strings(df, column):
    """
    Identify values containing non-uppercase, non-space characters.
    Useful for spotting typos, punctuation, or encoding issues.
    """
    condition = df[column].apply(
        lambda x: False if pd.isnull(x)
        else any(not (char.isupper() or char.isspace()) for char in str(x))
    )
    return df[condition][column].value_counts()

print('✅ Helper functions defined: clean_string, find_weird_strings')


#### 4.3a Country


In [ ]:
# --- Country ---
print(f'Unique countries before: {shark_clean.Country.nunique()}')
shark_clean['Country'] = shark_clean['Country'].apply(clean_string)

# Manual mapping for known ambiguous / multi-name entries
country_map = {
    'ST KITTS ? NEVIS': 'SAINT KITTS AND NEVIS',
    'ST. MARTIN': 'SAINT MARTIN',
    'ST. MAARTIN': 'SAINT MARTIN',
    'TURKS & CAICOS': 'TURKS AND CAICOS ISLANDS',
    'TRINIDAD & TOBAGO': 'TRINIDAD AND TOBAGO',
    'UNITED ARAB EMIRATES (UAE)': 'UNITED ARAB EMIRATES',
    'ST HELENA, BRITISH OVERSEAS TERRITORY': 'SAINT HELENA',
    'CEYLON (SRI LANKA)': 'SRI LANKA',
    'ANDAMAN / NICOBAR ISLANDS': 'INDIA',
    'SUDAN?': 'SUDAN',
    'MID-PACIFC OCEAN': None,
    'ASIA': None,
    'INDIAN OCEAN': None,
    'RED SEA': None,
}
shark_clean['Country'] = shark_clean['Country'].replace(country_map)

def clean_countries(x):
    """Remove ambiguous punctuation and geographic non-countries."""
    if pd.isna(x):
        return None
    x = x.replace('?', '').replace('.', '').replace('&', 'AND')
    if 'BETWEEN' in x:
        return None
    return x.split('/')[0].strip()

shark_clean['Country'] = shark_clean['Country'].apply(clean_countries)
print(f'Unique countries after: {shark_clean.Country.nunique()}')
shark_clean['Country'].value_counts().head(10)


#### 4.3b State & Location


In [ ]:
# --- State ---
print(f'Unique State values before: {shark_clean.State.nunique()}')
shark_clean['State'] = shark_clean['State'].apply(clean_string)
print(f'Unique State values after:  {shark_clean.State.nunique()}')

# --- Location ---
print(f'Unique Location values before: {shark_clean.Location.nunique()}')
shark_clean['Location'] = shark_clean['Location'].apply(clean_string)
print(f'Unique Location values after:  {shark_clean.Location.nunique()}')


#### 4.3c Fatality


In [ ]:
# --- Fatality ---
print(f'Unique Fatality values before: {shark_clean.Fatality.nunique()}')
shark_clean['Fatality'] = shark_clean['Fatality'].apply(clean_string)
print('Value counts after cleaning:')
print(shark_clean['Fatality'].value_counts())

# Rows with clearly erroneous Fatality values
weird_fatal = ['F', 'M', 'NQ', '2017', 'Y X 2']
print(f'\nRows with unexpected Fatality values: {shark_clean[shark_clean["Fatality"].isin(weird_fatal)].shape[0]}')


#### 4.3d Activity — Keyword Unification

~700 unique raw values unified into 10 standard categories via keyword matching.
Categories: FISHING · SWIMMING · SURFING · DIVING · BOATING · KAYAKING ·
STATIONARY · MARITIME ACCIDENT · OTHER · UNKNOWN


In [ ]:
# --- Activity ---
print(f'Unique Activity values before: {shark_clean.Activity.nunique()}')
shark_clean['Activity'] = shark_clean['Activity'].apply(clean_string)

# Keyword-based unification into standard categories
activity_map = {
    'fish': 'FISHING', 'swim': 'SWIMMING', 'bath': 'SWIMMING',
    'surf': 'SURFING', 'board': 'SURFING', 'div': 'DIVING',
    'ship': 'BOATING', 'sail': 'BOATING', 'boat': 'BOATING',
    'kayak': 'KAYAKING', 'canoe': 'KAYAKING',
    'wading': 'STATIONARY', 'stand': 'STATIONARY', 'tread': 'STATIONARY',
    'capsize': 'MARITIME ACCIDENT', 'sank': 'MARITIME ACCIDENT',
    'burn': 'MARITIME ACCIDENT', 'drop': 'MARITIME ACCIDENT',
    'explo': 'MARITIME ACCIDENT', 'fell': 'MARITIME ACCIDENT',
    'fall': 'MARITIME ACCIDENT',
}

def unify_activities(x):
    """Map raw Activity string to a standardized category using keyword matching."""
    if pd.isna(x):
        return 'UNKNOWN'
    x = str(x).lower()
    return next((label for key, label in activity_map.items() if key in x), 'OTHER')

shark_clean['Activity'] = shark_clean['Activity'].apply(unify_activities)
print(f'Unique Activity values after unification: {shark_clean.Activity.nunique()}')
shark_clean['Activity'].value_counts()


#### 4.3e Species — Keyword Unification

~300+ unique raw values unified into 15 named species + OTHER + UNKNOWN.
Named: WHITE SHARK · TIGER SHARK · BULL SHARK · HAMMERHEAD · BLACKTIP ·
MAKO · REEF SHARK · SAND SHARK · BLUE SHARK · NURSE SHARK · WOBBEGONG ·
LEMON SHARK · THRESHER SHARK


In [ ]:
# --- Species ---
print(f'Unique Species values before: {shark_clean.Species.nunique()}')
shark_clean['Species'] = shark_clean['Species'].apply(clean_string)

# Keyword-based species unification
shark_species_map = {
    'white': 'WHITE SHARK', 'great white': 'WHITE SHARK',
    'tiger': 'TIGER SHARK', 'bull': 'BULL SHARK',
    'hammer': 'HAMMERHEAD SHARK', 'blacktip': 'BLACKTIP SHARK',
    'mako': 'MAKO SHARK', 'reef': 'REEF SHARK',
    'coral reef': 'REEF SHARK', 'whitetip reef': 'REEF SHARK',
    'sand tiger': 'SAND SHARK', 'sandbar': 'SAND SHARK',
    'sand shark': 'SAND SHARK', 'blue': 'BLUE SHARK',
    'nurse': 'NURSE SHARK', 'wobbegong': 'WOBBEGONG SHARK',
    'lemon': 'LEMON SHARK', 'thresher': 'THRESHER SHARK',
}

def unify_species(x):
    """Map raw Species string to a standardized species name using keyword matching."""
    if pd.isna(x):
        return 'UNKNOWN'
    x = str(x).lower()
    return next((label for key, label in shark_species_map.items() if key in x), 'OTHER')

shark_clean['Species'] = shark_clean['Species'].apply(unify_species)
print(f'Unique Species values after unification: {shark_clean.Species.nunique()}')
shark_clean['Species'].value_counts()


---
### 4.4 Type Column — 5 Official GSAF Categories *(Irene)*

Fixes typos (`UNprovoked`, `Boatomg`) and normalizes casing.
Official categories: `Unprovoked`, `Provoked`, `Watercraft`, `Sea Disaster`, `Questionable`.


In [ ]:
# Fix typos and unify casing in the Type column
type_mapping = {
    'UNprovoked': 'Unprovoked', 'unprovoked': 'Unprovoked', 'UNPROVOKED': 'Unprovoked',
    'provoked': 'Provoked', 'PROVOKED': 'Provoked',
    'Boatomg': 'Watercraft',
    'Sea Disaster': 'Sea Disaster',
    'Invalid': 'Questionable',
}
shark_clean['Type'] = shark_clean['Type'].replace(type_mapping)

print('Type value counts after cleaning:')
print(shark_clean['Type'].value_counts(dropna=False))


---
### 4.5 Date Column — Multi-Pass Parsing *(Diana)*

The `Date` column contains 130 years of free-text entries in dozens of inconsistent formats.
A 4-step pipeline extracts clean `year_ext`, `month_ext`, and `Season` columns.

**Pipeline:**
1. `month_year()` — remove ordinal suffixes, normalize to 'Mon YYYY' format
2. `word_cleaner()` — strip noise words (Before, Circa, Reported…) from unparseable rows
3. `recog_daytimef()` — convert parseable strings to datetime objects
4. `ext_year_month()` — extract `year_ext` and `month_ext`, handle epoch artifacts
5. Derive `Season` from `month_ext` for H2 temporal analysis


In [ ]:
def month_year(val):
    """
    # Extracts month and year in int
    """
    if not isinstance(val, str):
        return val
    
    months = ["january","february","march","april","may","june",
              "july","august","september","October","november","december","jan","feb","mar","apr","may","jun",
              "jul","aug","sep","oct","nov","dec"]
    
    suffixes = ["st","nd","rd","th"]
    
    # Remove ordinal suffixes
    for s in suffixes:
        val = val.replace(s, "")
    
    val = val.strip()
    
    # Detect "MonthName DD" → flip to "DD MonthName"
    parts = val.lower().split()
    if len(parts) == 2:
        if parts[1].isdigit() and len(parts[1])<=2 and parts[0] in months:
            val = f"{parts[0]}"   # "was April 3, changed to april"
        elif parts[0].isdigit() and len(parts[0])<=2 and parts[1] in months:
            val = f"{parts[1]}"
        else:
            pass
            
    # Detect formats like "DD-MM-YYYY" 09-jun-2001 or 2001-jun-01
    parts = val.lower().replace('-',' ').split()
    if len(parts) == 3:
        #09-jun-2001
        if parts[1] in months and parts[2].isdigit() and len(parts[2])==4:
            val = f"{parts[1]} {parts[2]}"
        elif parts[0].isdigit() and len(parts[0])==4 and parts[1] in months:
            val = f"{parts[1]} {parts[0]}" #was 2001-jun-01 and becomes jun-2001
        else:
            pass

    # Detect formats like "YYYY- YYYY"
    if len(parts) == 2:
  
        if parts[0].isdigit() and parts[1].isdigit() and  len(parts[0])==4 and len(parts[1])==4:
            val = round((int(parts[0])+int(parts[1]))/2)
        elif parts[0].isdigit() and len(parts[0])==4 and parts[1] in months:
            val = f"{parts[1]} {parts[0]}" #was 2001-jun-01 and becomes jun-2001
        else:
            pass
    
    return val

def month_year_f(df: pd.DataFrame,col_date) ->pd.DataFrame:
    
    df1=df.copy()
    df1[col_date] = df1[col_date].apply(month_year)

    return df1

In [ ]:
from dateutil import parser 
def flex_dateparse(text):
   
    try:
        # fuzzy=True allows it to ignore words like "Reported" or "Ca.", "Late"

        return parser.parse(text, fuzzy=True)
    except:
        return None
        
def smart_cleaner(df,col_date):
    df1=df.copy()
    df1['fdate_check']=pd.to_datetime(df1[col_date], errors='coerce',dayfirst=True)
    mask= df1['fdate_check'].isnull()
    # Apply only to rows where the first pass failed
    df1.loc[mask, col_date] = df1.loc[mask,col_date].apply(flex_dateparse)
    df1=df1.drop('fdate_check',axis=1)
    return df1


In [ ]:
def word_cleaner(df,col_date, words_remove) ->pd.DataFrame:
    """
    df: The DataFrame
    target_col: The column you want to fix (e.g., 'Date')
    words_remove: List of strings to delete
    """
    df1=df.copy()
    df1['fdate_check']=pd.to_datetime(df1[col_date], errors='coerce',dayfirst=True)
    pattern='|'.join(words_remove)
    mask = df1['fdate_check'].isnull()
    df1.loc[mask,col_date]=df1.loc[mask,col_date].str.replace(pattern,'',regex=True).str.strip()
    df1=df1.drop('fdate_check',axis=1)
    return df1

In [ ]:
def recog_daytimef(df,col_date):
    """
## returns a copy of df which contains a new coloumn indicating NaT if the format if
#datetime cannot be determined. Also, returns a series of the rows from col_date for which NaN is present and which need fixing.
#errors='coerce' → if it can't parse, it writes NaT (= "unknown") instead of crashing
    """
    df1=df.copy()
    df1['fdate_check']=pd.to_datetime(df1[col_date], errors='coerce',dayfirst=True)
    mask = df1['fdate_check'].isnull()==False
    df1.loc[mask,'Date']=df1.loc[mask,'fdate_check']
    #df1= df1.drop('fdate_check', axis=1)
    return df1

In [ ]:
# Month name mapping
month_map = {
    'jan': 1, 'january': 1,
    'feb': 2, 'february': 2,
    'mar': 3, 'march': 3,
    'apr': 4, 'april': 4, 'ap':4,
    'may': 5,
    'jun': 6, 'june': 6,
    'jul': 7, 'july': 7,
    'aug': 8, 'august': 8, 'augu':8,
    'sep': 9, 'september': 9,
    'oct': 10, 'october': 10,
    'nov': 11, 'november': 11,
    'dec': 12, 'december': 12
}

def ext_year_month(date_val, fallback_year):
    date_str = str(date_val).strip().lower()

    # Trying direct datetime parse first ---
    parsed = pd.to_datetime(date_str, errors='coerce')
    
    # Excluding epoch artifacts (1970 with microsecond-encoded year)
    if parsed is not pd.NaT and parsed.year != 1970:
        return parsed.year, parsed.month

    # Month name only (e.g. "april", "jun") ---
    for name, num in month_map.items():
        if name in date_str:
            return fallback_year, num
        

    # Epoch artifact "1970-01-01 00:00:00.000001902"
    # microseconds field has the real year as last 4 digits
    if '1970' in date_str and '.' in date_str:
        micro_str = date_str.split('.')[-1]  # "000001902"
        year = micro_str[-4:]               # "1902"
        return int(year) if year.isdigit() else fallback_year, None

    return fallback_year, None

In [ ]:
# STEP 1 — Apply month_year normalisation
shark_clean = month_year_f(shark_clean, 'Date')
print(f'Unique Date values after step 1: {shark_clean.Date.nunique()}')
shark_clean['Date'].value_counts().head(10)


In [ ]:
# STEP 2 — Remove noise words from remaining unparseable dates
w_remove = [
    'Before', 'Reported', 'of', 'No date', 'Circa',
    'Prior to', 'After', 'or', 'late', 'A few years before',
    'Between', 'After Augu', 'before', 'Said to be'
]
shark_clean = word_cleaner(shark_clean, 'Date', w_remove)


In [ ]:
# STEP 3 — Convert parseable strings to datetime objects
shark_clean = recog_daytimef(shark_clean, 'Date')


In [ ]:
# STEP 4 — Extract year_ext and month_ext from cleaned Date
shark_clean[['year_ext', 'month_ext']] = shark_clean.apply(
    lambda row: pd.Series(ext_year_month(row['Date'], row['Year'])), axis=1
)

# Derive Season from month_ext (for H2 analysis)
season_map = {
    1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring',
    5: 'Spring', 6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn', 12: 'Winter'
}
shark_clean['Season'] = shark_clean['month_ext'].map(season_map)

print(f'Rows with unresolved year (year_ext == 0): {(shark_clean.year_ext == 0).sum()}')
shark_clean[['Date', 'Year', 'year_ext', 'month_ext', 'Season']].head(10)


---
### 4.6 Deduplication *(Both)*


In [ ]:
# Check for fully duplicate rows
n_dupes = shark_clean.duplicated().sum()
print(f'Fully duplicate rows: {n_dupes}')

if n_dupes > 0:
    shark_clean = shark_clean.drop_duplicates()
    print(f'✅ Duplicates removed. Shape now: {shark_clean.shape}')
else:
    print('✅ No duplicates found.')

print(f'\nFinal cleaned dataset: {shark_clean.shape[0]:,} rows × {shark_clean.shape[1]} columns')
shark_clean.dtypes


---
### 4.7 📌 Placeholder — Year / Numeric Type Fixes
> *To be completed: convert Year to Int64, validate year_ext range, handle extreme dates (pre-1900).*


In [ ]:
# 📌 PLACEHOLDER — Year dtype and range validation
# TODO: convert Year to Int64, filter or flag pre-1900 records
# shark_clean['Year'] = pd.to_numeric(shark_clean['Year'], errors='coerce').astype('Int64')


---
### 4.8 📌 Placeholder — Additional Transformations Before Analysis
> *Any remaining cleaning, feature engineering, or data shaping needed before EDA goes here.*


In [ ]:
# 📌 PLACEHOLDER — add any remaining transformations here
# e.g. filtering to modern era, binning years, creating derived columns


---
### ✅ Cleaning Summary

| Step | Column(s) | Technique | Status |
|------|-----------|-----------|--------|
| 4.1 | Multiple | Column dropping | ✅ Done |
| 4.2 | All | Column renaming / capitalize | ✅ Done |
| 4.3 | Country, State, Location, Fatality, Activity, Species | String cleaning + keyword mapping | ✅ Done |
| 4.4 | Type | Typo fix + category standardization | ✅ Done |
| 4.5 | Date | Multi-pass date parsing → year_ext, month_ext, Season | ✅ Done |
| 4.6 | All | Deduplication | ✅ Done |
| 4.7 | Year | Dtype conversion + range validation | 📌 Pending |
| 4.8 | TBD | Additional transformations for EDA | 📌 Pending |


---
### ✅ Cleaned Dataset — Quick Inspection


In [ ]:
# Final shape and column overview after full cleaning
print(f'Shape: {shark_clean.shape[0]:,} rows × {shark_clean.shape[1]} columns')
print(f'Columns: {list(shark_clean.columns)}')
shark_clean.dtypes


In [ ]:
shark_clean.head()


---
## 5️⃣ Exploratory Data Analysis (EDA)

With the cleaned dataset, we now explore distributions and patterns to form hypotheses.

> 📌 **Status:** EDA section to be completed after EDA module.
> Placeholder cells below — add analysis code here.
> Columns ready for analysis: `Country`, `Type`, `Activity`, `Species`,
> `Fatality`, `year_ext`, `month_ext`, `Season`.


In [ ]:
# 📌 PLACEHOLDER — incident type distribution
# shark_clean['Type'].value_counts()


In [ ]:
# 📌 PLACEHOLDER — top 10 activities
# shark_clean['Activity'].value_counts().head(10)


In [ ]:
# 📌 PLACEHOLDER — top 10 countries
# shark_clean['Country'].value_counts().head(10)


In [ ]:
# 📌 PLACEHOLDER — species distribution
# shark_clean['Species'].value_counts()


In [ ]:
# 📌 PLACEHOLDER — seasonal distribution (uses Season / month_ext from 4.5)
# shark_clean['Season'].value_counts()


In [ ]:
# 📌 PLACEHOLDER — incidents over time
# shark_clean['year_ext'].value_counts().sort_index()


---
## 6️⃣ Hypothesis Validation

We test the following hypotheses using aggregation and filtering:

### 🔬 Hypothesis 1 — Geographic Hotspots
> **H1:** Shark incidents (provoked and unprovoked) are more frequently recorded in specific
> coastal regions, indicating that certain areas are structurally higher-risk due to
> environmental or ecological conditions.
>
> *Status: 📌 To be validated in EDA phase*


In [ ]:
# 📌 PLACEHOLDER — H1: Geographic Hotspots
# top_countries = shark_clean['Country'].value_counts(dropna=True).head(10)
# pct_top5 = top_countries.head(5).sum() / len(shark_clean.dropna(subset=['Country'])) * 100
# print(top_countries)
# print(f'Top 5 countries = {pct_top5:.1f}% of all incidents')


### 🔬 Hypothesis 2 — Seasonal / Temporal Patterns
> **H2:** Shark incidents are more common during specific times of year, suggesting that seasonal
> behavioural patterns (mating, migration, feeding cycles) influence the likelihood of encounters.
>
> *Status: 📌 To be validated in EDA phase — requires year_ext / month_ext columns from 4.5*


In [ ]:
# 📌 PLACEHOLDER — H2: Seasonal Patterns
# Uses year_ext / month_ext / Season columns derived in 4.5
# shark_clean['Season'].value_counts()
# shark_clean['month_ext'].value_counts().sort_index()


### 🔬 Hypothesis 3 — Species: Variety vs. Quantity
> **H3:** Is a more varied sample of shark species preferable to a large volume of a single type?
> Testing whether species diversity correlates with incident patterns.
>
> *Status: 📌 To be validated — requires Species column from 4.3*


In [ ]:
# 📌 PLACEHOLDER — H3: Species Variety vs. Quantity
# shark_clean['Species'].value_counts()
# known = shark_clean[shark_clean['Species'] != 'UNKNOWN']
# print(f'Identified species records: {len(known)} of {len(shark_clean)}')


---
## 7️⃣ Aggregation & Pivot Tables

In [ ]:
# 📌 PLACEHOLDER — Country × Type pivot table
# top8 = shark_clean['Country'].value_counts(dropna=True).head(8).index
# pivot = pd.pivot_table(
#     shark_clean[shark_clean['Country'].isin(top8)],
#     index='Country', columns='Type',
#     values='Season', aggfunc='count', fill_value=0
# )
# pivot


---
## 8️⃣ Key Insights & Conclusions

> ⚠️ **Work in Progress** — this section will be completed once EDA and hypothesis validation are done.

**Planned conclusions structure:**
1. **H1 — Geography:** *[To be filled after country/region aggregation]*
2. **H2 — Seasonality:** *[To be filled after month/season distribution analysis]*
3. **H3 — Species:** *[To be filled after species frequency + diversity analysis]*
4. **Data quality notes:** Heavy cleaning required — Date, Type, Country, Activity, Species.
   Approximately X% of rows had missing or unparseable dates (handled via year_ext column).

---
*Notebook by Diana Carolina Yule Burbano & Irene Fafian — Ironhack Data Analytics Bootcamp, Week 2*


In [ ]:
# End of notebook — cleaning complete, analysis pending
print(f'Cleaned dataset ready: {shark_clean.shape[0]:,} rows × {shark_clean.shape[1]} columns')
print(f'Columns: {list(shark_clean.columns)}')
